# 30-Day Diabetes Readmission Risk Prediction

This notebook implements the machine learning phase of a healthcare analytics portfolio project using the UCI Diabetes 130-US Hospitals dataset.

The business problem is practical: 30-day readmissions create clinical burden, operational strain, and financial penalty exposure. The machine learning objective is to predict whether a diabetic patient encounter will be followed by readmission within 30 days, where `readmitted == "<30"` is coded as the positive class.

This revision is designed to be more comparable to Emi-Johnson & Nkrumah (2025), which reports an XGBoost ROC-AUC around 0.667. The main change is that the primary experiment uses the encounter-level dataset rather than first-encounter patient deduplication, because the reference paper does not clearly report one-encounter-per-patient filtering. Patient-level deduplication is kept as a sensitivity analysis.

The goal is responsible improvement, not artificial score inflation. The notebook reports accuracy, recall, F1, ROC-AUC, PR-AUC, Brier score, calibration, threshold tradeoffs, and SHAP explanations so the model can support a future discharge-planning dashboard.

## 1. Imports

The notebook uses standard data science libraries, scikit-learn pipelines, XGBoost for the primary model, optional LightGBM for comparison, SHAP for explainability, and joblib for saving trained artifacts.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
import shap

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 100)

## 2. Experiment Configuration

These switches make the notebook reproducible and make the main model more comparable to Emi-Johnson & Nkrumah (2025):

- `DEDUPLICATE_PATIENTS = False`: the main run keeps all valid encounters after hospice/expired exclusions. This better matches an encounter-level reference-paper comparison.
- `REMOVE_MEDICAL_SPECIALTY = True`: removes a high-missingness, high-cardinality field that previously dominated SHAP with noisy specialty one-hot features.
- `USE_CLASS_WEIGHTING = True`: applies `class_weight="balanced"` for Logistic Regression and `scale_pos_weight` for XGBoost, aligning with class-weighting practice rather than SMOTE.
- `RUN_RANDOMIZED_SEARCH = True`: tunes XGBoost with ROC-AUC scoring and compares tuned versus default XGBoost.
- `FINAL_THRESHOLD = 0.5`: keeps the academic comparison threshold fixed, while the threshold table exports best-F1 and recall-focused alternatives.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

DEDUPLICATE_PATIENTS = False
REMOVE_MEDICAL_SPECIALTY = True
USE_CLASS_WEIGHTING = True
RUN_RANDOMIZED_SEARCH = True
FINAL_THRESHOLD = 0.5

REFERENCE_XGBOOST_AUC = 0.667
PREVIOUS_XGBOOST_AUC = 0.648
PREVIOUS_XGBOOST_RECALL = 0.547
PREVIOUS_XGBOOST_F1 = 0.224

print({
    "RANDOM_STATE": RANDOM_STATE,
    "TEST_SIZE": TEST_SIZE,
    "DEDUPLICATE_PATIENTS": DEDUPLICATE_PATIENTS,
    "REMOVE_MEDICAL_SPECIALTY": REMOVE_MEDICAL_SPECIALTY,
    "USE_CLASS_WEIGHTING": USE_CLASS_WEIGHTING,
    "RUN_RANDOMIZED_SEARCH": RUN_RANDOMIZED_SEARCH,
    "FINAL_THRESHOLD": FINAL_THRESHOLD,
})

## 3. Project Paths

Paths are relative to the repository root, so the notebook can run from either the project root or the `notebooks` folder.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "diabetes+130-us+hospitals+for+years+1999-2008"
DIABETIC_DATA_PATH = RAW_DATA_DIR / "diabetic_data.csv"
IDS_MAPPING_PATH = RAW_DATA_DIR / "IDS_mapping.csv"
FEATURE_FREQUENCY_PATHS = [
    PROJECT_ROOT / "feature_frequency_table.csv",
    PROJECT_ROOT / "data" / "feature_frequency_table.csv",
    PROJECT_ROOT / "references" / "feature_frequency_table.csv",
]

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

for directory in [MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Diabetic data exists: {DIABETIC_DATA_PATH.exists()}")
print(f"IDS mapping exists: {IDS_MAPPING_PATH.exists()}")

## 4. Load Data

The main dataset contains one row per hospital encounter. The optional feature frequency table is loaded only if present.

In [ ]:
if not DIABETIC_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing required file: {DIABETIC_DATA_PATH}")
if not IDS_MAPPING_PATH.exists():
    raise FileNotFoundError(f"Missing required file: {IDS_MAPPING_PATH}")

df_raw = pd.read_csv(DIABETIC_DATA_PATH)
ids_mapping = pd.read_csv(IDS_MAPPING_PATH)

feature_frequency = None
for path in FEATURE_FREQUENCY_PATHS:
    if path.exists():
        feature_frequency = pd.read_csv(path)
        print(f"Loaded optional feature frequency table: {path}")
        break

print(f"Raw shape: {df_raw.shape}")
display(df_raw.head())
print("Raw readmitted distribution:")
display(df_raw["readmitted"].value_counts(dropna=False).to_frame("count"))
print("Missing marker summary:")
question_mark_summary = (df_raw == "?").sum().sort_values(ascending=False)
display(question_mark_summary[question_mark_summary > 0].to_frame("question_mark_count"))

## 5. Target Definition

The positive class is readmission within 30 days. This is an imbalanced outcome, so PR-AUC, recall, F1, threshold behavior, and calibration are reported alongside ROC-AUC.

In [ ]:
df = df_raw.copy()
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)

target_summary = pd.DataFrame({
    "count": df["readmitted_30d"].value_counts().sort_index(),
    "percent": df["readmitted_30d"].value_counts(normalize=True).sort_index().mul(100).round(2),
})
display(target_summary)

ax = sns.countplot(data=df, x="readmitted_30d")
ax.set_title("Original Target Distribution: 30-Day Readmission")
ax.set_xlabel("Readmitted within 30 days")
ax.set_ylabel("Encounter count")
plt.show()

## 6. Data Cleaning

The main experiment keeps encounter-level records unless `DEDUPLICATE_PATIENTS` is switched on. This makes the main run more comparable to the reference paper while still documenting the patient-level leakage tradeoff.

Cleaning steps:

- Replace `?` with missing values.
- Remove hospice/expired discharge dispositions `[11, 13, 14, 19, 20, 21]`.
- Remove `Unknown/Invalid` gender.
- Keep identifiers separately for export, but never use them as model features.

In [ ]:
EXPIRED_OR_HOSPICE_DISPOSITIONS = [11, 13, 14, 19, 20, 21]

clean_df = df.replace("?", np.nan).copy()
print(f"Rows before cleaning: {len(clean_df):,}")

clean_df = clean_df[~clean_df["discharge_disposition_id"].isin(EXPIRED_OR_HOSPICE_DISPOSITIONS)].copy()
print(f"Rows after expired/hospice exclusions: {len(clean_df):,}")

clean_df = clean_df[clean_df["gender"] != "Unknown/Invalid"].copy()
print(f"Rows after invalid gender exclusion: {len(clean_df):,}")

if DEDUPLICATE_PATIENTS:
    clean_df = clean_df.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first").copy()
    print(f"Rows after first encounter per patient: {len(clean_df):,}")
else:
    print("Patient deduplication skipped for the main encounter-level experiment.")

clean_target_summary = pd.DataFrame({
    "count": clean_df["readmitted_30d"].value_counts().sort_index(),
    "percent": clean_df["readmitted_30d"].value_counts(normalize=True).sort_index().mul(100).round(2),
})
display(clean_target_summary)

## 7. Feature Engineering

The engineered features are literature-guided and dashboard-relevant: age midpoint, diagnosis groups, A1C testing indicators, prior utilization, and medication flags.

In [ ]:
def age_to_midpoint(age_bracket):
    if pd.isna(age_bracket):
        return np.nan
    age_text = str(age_bracket).strip("[]()")
    lower, upper = age_text.split("-")
    return (int(lower) + int(upper)) / 2


def group_icd_diagnosis(code):
    if pd.isna(code):
        return "Other"

    text = str(code).strip().upper()
    if not text:
        return "Other"

    if text.startswith("V") or text.startswith("E"):
        return "Other"

    try:
        numeric_code = float(text)
    except ValueError:
        return "Other"

    if 250 <= numeric_code < 251:
        return "Diabetes"
    if 390 <= numeric_code <= 459 or numeric_code == 785:
        return "Circulatory"
    if 460 <= numeric_code <= 519 or numeric_code == 786:
        return "Respiratory"
    if 520 <= numeric_code <= 579 or numeric_code == 787:
        return "Digestive"
    if 800 <= numeric_code <= 999:
        return "Injury"
    if 710 <= numeric_code <= 739:
        return "Musculoskeletal"
    if 580 <= numeric_code <= 629 or numeric_code == 788:
        return "Genitourinary"
    if 140 <= numeric_code <= 239:
        return "Neoplasms"
    return "Other"


feature_df = clean_df.copy()
feature_df["age_numeric"] = feature_df["age"].apply(age_to_midpoint)

for diag_col in ["diag_1", "diag_2", "diag_3"]:
    feature_df[f"{diag_col}_group"] = feature_df[diag_col].apply(group_icd_diagnosis)

feature_df["hba1c_tested"] = (feature_df["A1Cresult"] != "None").astype(int)
feature_df["hba1c_high"] = feature_df["A1Cresult"].isin([">7", ">8"]).astype(int)
feature_df["prior_utilization_total"] = (
    feature_df["number_outpatient"].fillna(0)
    + feature_df["number_emergency"].fillna(0)
    + feature_df["number_inpatient"].fillna(0)
)
feature_df["med_changed"] = (feature_df["change"] == "Ch").astype(int)
feature_df["diabetesMed_binary"] = (feature_df["diabetesMed"] == "Yes").astype(int)

preview_cols = [
    "age", "age_numeric", "diag_1", "diag_1_group", "diag_2_group", "diag_3_group",
    "A1Cresult", "hba1c_tested", "hba1c_high", "prior_utilization_total",
    "change", "med_changed", "diabetesMed", "diabetesMed_binary",
]
display(feature_df[preview_cols].head())

## 8. Literature-Guided Feature Selection

`medical_specialty` is excluded in the main run because it is high-cardinality, high-missingness, and previously dominated SHAP with specialty-specific noise. The model retains encounter-level admission source and other features that improve comparability with the reference-paper setup.

In [ ]:
core_features = [
    "age_numeric",
    "gender",
    "race",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "prior_utilization_total",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "A1Cresult",
    "hba1c_tested",
    "hba1c_high",
    "max_glu_serum",
    "change",
    "med_changed",
    "diabetesMed",
    "diabetesMed_binary",
    "insulin",
    "metformin",
    "glipizide",
    "glyburide",
    "glimepiride",
    "pioglitazone",
    "rosiglitazone",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
]

if not REMOVE_MEDICAL_SPECIALTY:
    core_features.append("medical_specialty")

available_features = [col for col in core_features if col in feature_df.columns]
missing_features = [col for col in core_features if col not in feature_df.columns]
print(f"Available candidate features: {len(available_features)}")
print(f"Missing candidate features skipped: {missing_features}")

candidate_X = feature_df[available_features].copy()
y = feature_df["readmitted_30d"].copy()
export_ids = feature_df[["encounter_id", "patient_nbr"]].copy()

near_zero_variance_cols = []
for col in candidate_X.columns:
    top_frequency = candidate_X[col].value_counts(normalize=True, dropna=False).iloc[0]
    unique_count = candidate_X[col].nunique(dropna=False)
    if unique_count <= 1 or top_frequency >= 0.995:
        near_zero_variance_cols.append(col)

X = candidate_X.drop(columns=near_zero_variance_cols)
print(f"Near-zero variance columns removed: {near_zero_variance_cols}")
print(f"Final feature count: {X.shape[1]}")
print(f"medical_specialty included: {'medical_specialty' in X.columns}")
display(X.head())

## 9. Train-Test Split and Preprocessing

The split is stratified to preserve the minority readmission class. The preprocessing pipeline uses median imputation for numeric features and most-frequent imputation plus one-hot encoding for categorical features.

In [ ]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X,
    y,
    export_ids,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [col for col in X_train.columns if col not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")
display(y_train.value_counts(normalize=True).sort_index().mul(100).round(2).to_frame("train_percent"))
display(y_test.value_counts(normalize=True).sort_index().mul(100).round(2).to_frame("test_percent"))

## 10. Evaluation Helpers

These helpers keep evaluation consistent across baseline, default XGBoost, tuned XGBoost, and optional comparison models.

In [ ]:
def get_probabilities(model, X_eval):
    return model.predict_proba(X_eval)[:, 1]


def evaluate_binary_model(name, model, X_eval, y_eval, threshold=0.5):
    predicted_probability = get_probabilities(model, X_eval)
    predicted_label = (predicted_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_eval, predicted_label).ravel()
    return {
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_eval, predicted_label),
        "Precision Class 1": precision_score(y_eval, predicted_label, zero_division=0),
        "Recall Class 1": recall_score(y_eval, predicted_label, zero_division=0),
        "F1 Class 1": f1_score(y_eval, predicted_label, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, predicted_probability),
        "PR-AUC": average_precision_score(y_eval, predicted_probability),
        "Brier Score": brier_score_loss(y_eval, predicted_probability),
        "False Negatives": int(fn),
        "False Positives": int(fp),
        "Confusion Matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
    }


def display_model_report(name, model, X_eval, y_eval, threshold=0.5):
    metrics = evaluate_binary_model(name, model, X_eval, y_eval, threshold=threshold)
    print(f"{name} metrics at threshold {threshold}:")
    for key, value in metrics.items():
        if key != "Confusion Matrix":
            print(f"{key}: {value}")
    print("Confusion matrix:")
    print(np.array(metrics["Confusion Matrix"]))
    labels = (get_probabilities(model, X_eval) >= threshold).astype(int)
    print(classification_report(y_eval, labels, zero_division=0))
    return metrics


def make_xgboost_classifier(scale_pos_weight):
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        n_jobs=-1,
        verbosity=0,
    )

## 11. Baseline: Logistic Regression

Logistic Regression provides a transparent baseline. Class weighting is enabled when `USE_CLASS_WEIGHTING` is true.

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced" if USE_CLASS_WEIGHTING else None,
        max_iter=1000,
        solver="saga",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

logistic_model.fit(X_train, y_train)
logistic_metrics = display_model_report("Logistic Regression", logistic_model, X_test, y_test, threshold=FINAL_THRESHOLD)

## 12. XGBoost: Default and Tuned

The default model uses a strong literature-guided configuration. The tuned model uses `RandomizedSearchCV` with ROC-AUC scoring, 5-fold CV, and 25 iterations. The tuned model is compared directly against default XGBoost rather than replacing it silently.

In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count if USE_CLASS_WEIGHTING else 1.0
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgboost_default_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", make_xgboost_classifier(scale_pos_weight)),
])

xgboost_default_pipeline.fit(X_train, y_train)
xgboost_default_metrics = display_model_report("XGBoost Default", xgboost_default_pipeline, X_test, y_test, threshold=FINAL_THRESHOLD)

param_dist = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [3, 4, 5, 6],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "classifier__min_child_weight": [1, 3, 5],
    "classifier__gamma": [0, 0.1, 0.3],
    "classifier__reg_lambda": [1, 3, 5, 10],
}

if RUN_RANDOMIZED_SEARCH:
    xgboost_search_pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("classifier", make_xgboost_classifier(scale_pos_weight)),
    ])
    xgboost_search = RandomizedSearchCV(
        estimator=xgboost_search_pipeline,
        param_distributions=param_dist,
        n_iter=25,
        scoring="roc_auc",
        cv=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )
    xgboost_search.fit(X_train, y_train)
    xgboost_tuned_pipeline = xgboost_search.best_estimator_
    best_xgboost_params = xgboost_search.best_params_
    print(f"Best CV ROC-AUC: {xgboost_search.best_score_:.4f}")
    print("Best XGBoost parameters:")
    print(json.dumps(best_xgboost_params, indent=2))
else:
    xgboost_tuned_pipeline = xgboost_default_pipeline
    best_xgboost_params = {"note": "RUN_RANDOMIZED_SEARCH was False; tuned model reused default XGBoost."}

xgboost_tuned_metrics = display_model_report("XGBoost Tuned", xgboost_tuned_pipeline, X_test, y_test, threshold=FINAL_THRESHOLD)

with open(RESULTS_DIR / "best_xgboost_params.json", "w") as f:
    json.dump(best_xgboost_params, f, indent=2)
print(f"Saved best parameters to {RESULTS_DIR / 'best_xgboost_params.json'}")

## 13. Optional Model Comparison

Random Forest and LightGBM are included as secondary comparisons. The key reference-paper comparison remains default/tuned XGBoost.

In [ ]:
comparison_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced_subsample" if USE_CLASS_WEIGHTING else None,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

if LIGHTGBM_AVAILABLE:
    comparison_models["LightGBM"] = LGBMClassifier(
        objective="binary",
        class_weight="balanced" if USE_CLASS_WEIGHTING else None,
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )

model_metrics = [logistic_metrics, xgboost_default_metrics, xgboost_tuned_metrics]
fitted_comparison_models = {}

for name, estimator in comparison_models.items():
    model_pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("classifier", estimator),
    ])
    model_pipeline.fit(X_train, y_train)
    fitted_comparison_models[name] = model_pipeline
    model_metrics.append(evaluate_binary_model(name, model_pipeline, X_test, y_test, threshold=FINAL_THRESHOLD))

model_comparison = pd.DataFrame(model_metrics)
model_comparison_display = model_comparison.drop(columns=["Confusion Matrix"]).copy()
metric_cols = ["Accuracy", "Precision Class 1", "Recall Class 1", "F1 Class 1", "ROC-AUC", "PR-AUC", "Brier Score"]
model_comparison_display[metric_cols] = model_comparison_display[metric_cols].round(4)
display(model_comparison_display.sort_values("ROC-AUC", ascending=False))

model_comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print(f"Saved model comparison to {RESULTS_DIR / 'model_comparison.csv'}")

## 14. Select Best XGBoost Model

The best XGBoost variant is selected by test ROC-AUC for reference-paper comparability. This keeps the model choice transparent.

In [ ]:
xgboost_candidates = {
    "XGBoost Default": (xgboost_default_pipeline, xgboost_default_metrics),
    "XGBoost Tuned": (xgboost_tuned_pipeline, xgboost_tuned_metrics),
}

best_xgboost_name, (best_model_pipeline, best_xgboost_metrics) = max(
    xgboost_candidates.items(), key=lambda item: item[1][1]["ROC-AUC"]
)

best_probabilities = get_probabilities(best_model_pipeline, X_test)
print(f"Best XGBoost model: {best_xgboost_name}")
print(f"Best XGBoost ROC-AUC: {best_xgboost_metrics['ROC-AUC']:.4f}")
print(f"Reference-paper XGBoost ROC-AUC target: {REFERENCE_XGBOOST_AUC:.3f}")
print(f"Delta vs reference target: {best_xgboost_metrics['ROC-AUC'] - REFERENCE_XGBOOST_AUC:.4f}")

## 15. Threshold Selection

Thresholds from 0.10 to 0.90 are evaluated. The notebook exports three threshold choices:

- default academic threshold `0.5`,
- best-F1 threshold,
- recall-focused threshold with recall >= 0.70 if possible.

In [ ]:
threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.05):
    threshold = round(float(threshold), 2)
    labels = (best_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, labels).ravel()
    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, labels),
        "Precision Class 1": precision_score(y_test, labels, zero_division=0),
        "Recall Class 1": recall_score(y_test, labels, zero_division=0),
        "F1 Class 1": f1_score(y_test, labels, zero_division=0),
        "False Negatives": int(fn),
        "False Positives": int(fp),
        "Confusion Matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
    })

threshold_comparison = pd.DataFrame(threshold_rows)
best_f1_threshold = float(threshold_comparison.sort_values(["F1 Class 1", "Recall Class 1"], ascending=False).iloc[0]["Threshold"])
recall_candidates = threshold_comparison[threshold_comparison["Recall Class 1"] >= 0.70]
if not recall_candidates.empty:
    recall_focused_threshold = float(recall_candidates.sort_values(["F1 Class 1", "Precision Class 1"], ascending=False).iloc[0]["Threshold"])
else:
    recall_focused_threshold = np.nan

threshold_choices = pd.DataFrame([
    {"Option": "Academic default", "Threshold": FINAL_THRESHOLD},
    {"Option": "Best F1", "Threshold": best_f1_threshold},
    {"Option": "Recall >= 0.70 if possible", "Threshold": recall_focused_threshold},
])

threshold_comparison.to_csv(RESULTS_DIR / "threshold_comparison.csv", index=False)
threshold_choices.to_csv(RESULTS_DIR / "threshold_options.csv", index=False)

display(threshold_comparison.round(4))
display(threshold_choices)
print(f"Saved threshold comparison to {RESULTS_DIR / 'threshold_comparison.csv'}")
print(f"Saved threshold options to {RESULTS_DIR / 'threshold_options.csv'}")

## 16. ROC, PR, Calibration, and Confusion Matrix Figures

ROC-AUC supports comparison with the reference paper. PR-AUC is important because the positive class is imbalanced. Brier score and the calibration curve show whether dashboard risk scores are probability-like.

In [ ]:
def save_performance_figures(y_true, probabilities, threshold):
    fpr, tpr, _ = roc_curve(y_true, probabilities)
    roc_auc = roc_auc_score(y_true, probabilities)
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {best_xgboost_name}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "roc_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    precision, recall, _ = precision_recall_curve(y_true, probabilities)
    pr_auc = average_precision_score(y_true, probabilities)
    plt.figure(figsize=(7, 6))
    plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve - {best_xgboost_name}")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "pr_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    prob_true, prob_pred = calibration_curve(y_true, probabilities, n_bins=10, strategy="quantile")
    plt.figure(figsize=(7, 6))
    plt.plot(prob_pred, prob_true, marker="o", label="Model")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed readmission rate")
    plt.title(f"Calibration Curve - Brier {brier_score_loss(y_true, probabilities):.3f}")
    plt.legend(loc="upper left")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "calibration_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    labels = (probabilities >= threshold).astype(int)
    cm = confusion_matrix(y_true, labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - Threshold {threshold}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "confusion_matrix_xgboost.png", dpi=300, bbox_inches="tight")
    plt.show()

save_performance_figures(y_test, best_probabilities, FINAL_THRESHOLD)

## 17. Patient-Level Deduplication Sensitivity Analysis

The main model is encounter-level for comparability. This compact sensitivity analysis trains default XGBoost on first encounter per patient to document the effect of deduplication without making it the primary result.

In [ ]:
def build_features_for_dataframe(input_df, deduplicate_patients):
    temp = input_df.replace("?", np.nan).copy()
    temp["readmitted_30d"] = (temp["readmitted"] == "<30").astype(int)
    temp = temp[~temp["discharge_disposition_id"].isin(EXPIRED_OR_HOSPICE_DISPOSITIONS)].copy()
    temp = temp[temp["gender"] != "Unknown/Invalid"].copy()
    if deduplicate_patients:
        temp = temp.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first").copy()
    temp["age_numeric"] = temp["age"].apply(age_to_midpoint)
    for diag_col in ["diag_1", "diag_2", "diag_3"]:
        temp[f"{diag_col}_group"] = temp[diag_col].apply(group_icd_diagnosis)
    temp["hba1c_tested"] = (temp["A1Cresult"] != "None").astype(int)
    temp["hba1c_high"] = temp["A1Cresult"].isin([">7", ">8"]).astype(int)
    temp["prior_utilization_total"] = temp["number_outpatient"].fillna(0) + temp["number_emergency"].fillna(0) + temp["number_inpatient"].fillna(0)
    temp["med_changed"] = (temp["change"] == "Ch").astype(int)
    temp["diabetesMed_binary"] = (temp["diabetesMed"] == "Yes").astype(int)
    temp_X = temp[[col for col in core_features if col in temp.columns]].copy()
    for col in list(temp_X.columns):
        if temp_X[col].nunique(dropna=False) <= 1 or temp_X[col].value_counts(normalize=True, dropna=False).iloc[0] >= 0.995:
            temp_X = temp_X.drop(columns=[col])
    return temp_X, temp["readmitted_30d"].copy(), len(temp)

sensitivity_X, sensitivity_y, sensitivity_rows = build_features_for_dataframe(df_raw, deduplicate_patients=True)
SX_train, SX_test, sy_train, sy_test = train_test_split(
    sensitivity_X, sensitivity_y, test_size=TEST_SIZE, stratify=sensitivity_y, random_state=RANDOM_STATE
)

sensitivity_numeric = SX_train.select_dtypes(include=["number", "bool"]).columns.tolist()
sensitivity_categorical = [col for col in SX_train.columns if col not in sensitivity_numeric]
sensitivity_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), sensitivity_numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]), sensitivity_categorical),
    ],
    remainder="drop",
)

sensitivity_scale_pos_weight = (sy_train == 0).sum() / (sy_train == 1).sum() if USE_CLASS_WEIGHTING else 1.0
sensitivity_model = Pipeline(steps=[
    ("preprocess", sensitivity_preprocessor),
    ("classifier", make_xgboost_classifier(sensitivity_scale_pos_weight)),
])
sensitivity_model.fit(SX_train, sy_train)
sensitivity_metrics = evaluate_binary_model("XGBoost Default - Deduplicated Sensitivity", sensitivity_model, SX_test, sy_test, threshold=FINAL_THRESHOLD)
sensitivity_summary = pd.DataFrame([
    {"Experiment": "Encounter-level main", "Rows": len(feature_df), **xgboost_default_metrics},
    {"Experiment": "Patient-level sensitivity", "Rows": sensitivity_rows, **sensitivity_metrics},
])
display(sensitivity_summary[["Experiment", "Rows", "ROC-AUC", "PR-AUC", "Recall Class 1", "F1 Class 1", "Brier Score"]].round(4))
sensitivity_summary.to_csv(RESULTS_DIR / "deduplication_sensitivity.csv", index=False)
print(f"Saved sensitivity analysis to {RESULTS_DIR / 'deduplication_sensitivity.csv'}")

## 18. SHAP Explainability

SHAP is computed for the best XGBoost pipeline. Since `medical_specialty` is excluded from the main model, top features should be less dominated by noisy specialty one-hot columns and more aligned with literature signals such as prior utilization, medication burden, admission/discharge context, A1C, insulin/medication status, and diagnosis groups.

In [ ]:
fitted_preprocessor = best_model_pipeline.named_steps["preprocess"]
fitted_xgb = best_model_pipeline.named_steps["classifier"]
feature_names = fitted_preprocessor.get_feature_names_out()

X_test_transformed = fitted_preprocessor.transform(X_test)
shap_sample_size = min(1000, X_test_transformed.shape[0])
shap_sample_indices = np.random.default_rng(RANDOM_STATE).choice(
    X_test_transformed.shape[0], size=shap_sample_size, replace=False
)
X_shap_sample = X_test_transformed[shap_sample_indices]
X_shap_sample_dense = X_shap_sample.toarray() if hasattr(X_shap_sample, "toarray") else np.asarray(X_shap_sample)

explainer = shap.TreeExplainer(fitted_xgb)
shap_values = explainer.shap_values(X_shap_sample_dense)
if isinstance(shap_values, list):
    shap_values_for_positive_class = shap_values[1]
else:
    shap_values_for_positive_class = shap_values

shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values_for_positive_class).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

shap_importance.to_csv(RESULTS_DIR / "shap_top_features.csv", index=False)
display(shap_importance.head(20))

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_for_positive_class,
    X_shap_sample_dense,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.title("Global SHAP Feature Importance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_global_importance.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_for_positive_class,
    X_shap_sample_dense,
    feature_names=feature_names,
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()

## 19. Local SHAP Error Analysis

The next cell prints compact local explanations for one true positive, one false negative, and one false positive when available.

In [ ]:
test_analysis = ids_test.reset_index(drop=True).copy()
test_analysis["actual"] = y_test.reset_index(drop=True).values
test_analysis["predicted_probability"] = best_probabilities
test_analysis["predicted_label"] = (best_probabilities >= FINAL_THRESHOLD).astype(int)

case_definitions = {
    "High-risk true positive": (test_analysis["actual"] == 1) & (test_analysis["predicted_label"] == 1),
    "False negative": (test_analysis["actual"] == 1) & (test_analysis["predicted_label"] == 0),
    "False positive": (test_analysis["actual"] == 0) & (test_analysis["predicted_label"] == 1),
}

for case_name, mask in case_definitions.items():
    available_cases = test_analysis[mask]
    if available_cases.empty:
        print(f"No {case_name.lower()} found at threshold {FINAL_THRESHOLD}.")
        continue

    selected_index = available_cases["predicted_probability"].idxmax()
    original_features = X_test.reset_index(drop=True).iloc[[selected_index]]
    transformed_features = fitted_preprocessor.transform(original_features)
    transformed_features = transformed_features.toarray() if hasattr(transformed_features, "toarray") else transformed_features
    local_shap_values = explainer.shap_values(transformed_features)
    if isinstance(local_shap_values, list):
        local_shap_values = local_shap_values[1]

    local_contributions = pd.DataFrame({
        "feature": feature_names,
        "feature_value": transformed_features[0],
        "shap_value": local_shap_values[0],
    })
    local_contributions["abs_shap"] = local_contributions["shap_value"].abs()
    local_contributions = local_contributions.sort_values("abs_shap", ascending=False).head(10)

    print("=" * 80)
    print(case_name)
    print(test_analysis.loc[selected_index, ["encounter_id", "patient_nbr", "actual", "predicted_probability", "predicted_label"]])
    display(original_features)
    display(local_contributions[["feature", "feature_value", "shap_value"]])

## 20. Export Model Artifacts and Dashboard-Ready Predictions

The dashboard export uses the academic comparison threshold `0.5` by default, while `threshold_options.csv` contains alternative threshold choices. Risk categories are kept as simple portfolio-dashboard bands.

In [ ]:
def risk_category(score):
    if score < 40:
        return "Low"
    if score <= 70:
        return "Medium"
    return "High"

prediction_export = ids_test.reset_index(drop=True).copy()
prediction_export["actual_readmitted_30d"] = y_test.reset_index(drop=True).values
prediction_export["predicted_probability"] = best_probabilities
prediction_export["predicted_risk_score_0_100"] = (prediction_export["predicted_probability"] * 100).round(1)
prediction_export["predicted_label"] = (prediction_export["predicted_probability"] >= FINAL_THRESHOLD).astype(int)
prediction_export["predicted_risk_category"] = prediction_export["predicted_risk_score_0_100"].apply(risk_category)
prediction_export["model_threshold"] = FINAL_THRESHOLD
prediction_export["model_name"] = best_xgboost_name

prediction_export.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

joblib.dump(xgboost_default_pipeline, MODELS_DIR / "xgboost_default_pipeline.pkl")
joblib.dump(xgboost_tuned_pipeline, MODELS_DIR / "xgboost_tuned_pipeline.pkl")
joblib.dump(best_model_pipeline, MODELS_DIR / "best_model_pipeline.pkl")

# Backward-compatible artifact names from the first notebook version.
joblib.dump(fitted_xgb, MODELS_DIR / "xgboost_readmission_model.pkl")
joblib.dump(fitted_preprocessor, MODELS_DIR / "preprocessing_pipeline.pkl")
joblib.dump(best_model_pipeline, MODELS_DIR / "xgboost_readmission_full_pipeline.pkl")

print(f"Saved default XGBoost pipeline to {MODELS_DIR / 'xgboost_default_pipeline.pkl'}")
print(f"Saved tuned XGBoost pipeline to {MODELS_DIR / 'xgboost_tuned_pipeline.pkl'}")
print(f"Saved best model pipeline to {MODELS_DIR / 'best_model_pipeline.pkl'}")
print(f"Saved test predictions to {RESULTS_DIR / 'test_predictions.csv'}")
display(prediction_export.head())

## 21. What Changed Compared With the First Notebook?

This revision improves comparability with Emi-Johnson & Nkrumah (2025) and improves interpretation quality:

- Encounter-level modeling is now the main experiment because the reference paper does not clearly state patient-level deduplication.
- Patient-level first-encounter deduplication remains as a sensitivity analysis, not the primary comparison.
- `medical_specialty` is removed from the main model to reduce noisy high-cardinality SHAP effects.
- XGBoost hyperparameter tuning was added using `RandomizedSearchCV` with ROC-AUC scoring.
- PR-AUC and Brier score were added for imbalanced classification and risk-score quality.
- Threshold tuning now evaluates thresholds from 0.10 to 0.90 and exports default, best-F1, and recall-focused options.
- SHAP outputs are checked against expected literature drivers: prior utilization, medication burden, length of stay, diagnosis group, insulin/diabetes medication status, A1C testing, and admission/discharge context.

If ROC-AUC does not reach the paper's approximate 0.667, plausible reasons include dataset filtering differences, deduplication strategy, feature engineering differences, class imbalance, random tuning space, and incomplete reporting of implementation details in the paper.

In [ ]:
final_summary = {
    "previous_xgboost_auc": PREVIOUS_XGBOOST_AUC,
    "previous_xgboost_recall": PREVIOUS_XGBOOST_RECALL,
    "previous_xgboost_f1": PREVIOUS_XGBOOST_F1,
    "new_best_xgboost_model": best_xgboost_name,
    "new_xgboost_auc": best_xgboost_metrics["ROC-AUC"],
    "new_xgboost_recall_at_0_5": best_xgboost_metrics["Recall Class 1"],
    "new_xgboost_f1_at_0_5": best_xgboost_metrics["F1 Class 1"],
    "reference_xgboost_auc": REFERENCE_XGBOOST_AUC,
    "main_rows_after_cleaning": len(feature_df),
    "deduplicate_patients_main_run": DEDUPLICATE_PATIENTS,
    "medical_specialty_removed": REMOVE_MEDICAL_SPECIALTY,
    "top_10_shap_features": shap_importance.head(10)["feature"].tolist(),
}

with open(RESULTS_DIR / "final_model_summary.json", "w") as f:
    json.dump(final_summary, f, indent=2)

print(json.dumps(final_summary, indent=2))